# Data Career Navigator - Your Career in the Age of AI

An open-source, evidence-based career-transition agent for data & AI professionals.

It answers two questions:

1. **Career Navigator** - *Given my history and where the AI job market is heading, what should my next role be?*
2. **JD Analyzer** - *For this job description, how much can AI take over, and what stays human?*

**Design principle:** career advice is treated as a **data problem**, not a black-box chatbot answer. Every role is a **skill vector**, and every score is computed with transparent math you can inspect. The product is framed around **career evolution, not job replacement**.

**Runs fully offline** - no API key required. An optional LLM layer (bottom of the notebook) can add richer text.

> **Google Colab:** just upload this single `.ipynb` and run all cells. The first cell writes the engine and data into the runtime automatically - you do **not** need to upload any other files.

---

## 0. Setup (self-contained)

In [ ]:
# =====================================================================
# SELF-CONTAINED SETUP (Colab-ready)
# ---------------------------------------------------------------------
# This cell installs deps and writes the engine + data into the runtime,
# so you can upload ONLY this .ipynb to Colab and run it top-to-bottom.
# No need to upload the src/ or data/ folders separately.
# =====================================================================
import subprocess, sys, os

# 1) Install the only two required dependencies (quietly).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas'], check=False)

# 2) Materialize the project files into the current runtime.
os.makedirs('data', exist_ok=True)
os.makedirs('src', exist_ok=True)

_FILES = {
    'data/skills.json': '{\n  "_note": "Canonical skill dimensions used to build role vectors. Values in roles.json are 0-100 proficiency expectations for that role. Categories help group skills and identify transferable vs technical skills.",\n  "skills": [\n    { "id": "sql",                "label": "SQL",                      "category": "data" },\n    { "id": "data_modeling",      "label": "Data Modeling",            "category": "data" },\n    { "id": "bi_reporting",       "label": "BI & Reporting",           "category": "data" },\n    { "id": "data_viz",           "label": "Data Visualization",       "category": "data" },\n    { "id": "statistics",         "label": "Statistics",               "category": "analytics" },\n    { "id": "experimentation",    "label": "Experimentation / A-B",    "category": "analytics" },\n    { "id": "python",             "label": "Python",                   "category": "engineering" },\n    { "id": "software_eng",       "label": "Software Engineering",     "category": "engineering" },\n    { "id": "git_cicd",           "label": "Git / CI-CD",              "category": "engineering" },\n    { "id": "cloud",              "label": "Cloud Platforms",          "category": "engineering" },\n    { "id": "data_engineering",   "label": "Data Engineering / ETL",   "category": "engineering" },\n    { "id": "dbt_semantic",       "label": "dbt / Semantic Layer",     "category": "engineering" },\n    { "id": "machine_learning",   "label": "Machine Learning",         "category": "ai" },\n    { "id": "mlops",              "label": "MLOps",                    "category": "ai" },\n    { "id": "genai_llm",          "label": "GenAI / LLMs",             "category": "ai" },\n    { "id": "rag",                "label": "RAG / Vector Search",      "category": "ai" },\n    { "id": "ai_agents",          "label": "AI Agents",                "category": "ai" },\n    { "id": "ai_evaluation",      "label": "AI Evaluation",            "category": "ai" },\n    { "id": "prompt_eng",         "label": "Prompt Engineering",       "category": "ai" },\n    { "id": "business_analysis",  "label": "Business Analysis",        "category": "business" },\n    { "id": "stakeholder_mgmt",   "label": "Stakeholder Management",   "category": "business" },\n    { "id": "problem_framing",    "label": "Problem Framing",          "category": "business" },\n    { "id": "product_mgmt",       "label": "Product Management",       "category": "business" },\n    { "id": "domain_knowledge",   "label": "Domain / Industry Knowledge", "category": "business" }\n  ],\n  "transferable_categories": ["business", "analytics"],\n  "technical_categories": ["data", "engineering", "ai"]\n}\n',
    'data/roles.json': '{\n  "_note": "Curated SEED data. Each role is a skill vector (0-100 expected proficiency) plus market signals. Scores are illustrative and computed transparently. Replace with O*NET + BLS + open job-postings data for authoritative numbers. \'demand\' = current market demand 0-100, \'growth\' = projected growth 0-100, \'remote_share\' = % of postings that are remote, \'ai_resilience\' = how much the role is augmented rather than automated 0-100, \'salary_index\' = relative salary 0-100 (illustrative), \'onet_code\' = closest O*NET-SOC occupation for grounding.",\n  "roles": [\n    {\n      "id": "data_analyst",\n      "title": "Data Analyst",\n      "family": "analytics",\n      "onet_code": "15-2051.00",\n      "seniority": "mid",\n      "vector": {\n        "sql": 90, "data_modeling": 55, "bi_reporting": 90, "data_viz": 85, "statistics": 70,\n        "experimentation": 45, "python": 55, "software_eng": 25, "git_cicd": 25, "cloud": 30,\n        "data_engineering": 30, "dbt_semantic": 30, "machine_learning": 30, "mlops": 10,\n        "genai_llm": 25, "rag": 10, "ai_agents": 10, "ai_evaluation": 10, "prompt_eng": 30,\n        "business_analysis": 80, "stakeholder_mgmt": 75, "problem_framing": 70, "product_mgmt": 35,\n        "domain_knowledge": 70\n      },\n      "demand": 78, "growth": 55, "remote_share": 62, "ai_resilience": 55, "salary_index": 55\n    },\n    {\n      "id": "senior_data_analyst",\n      "title": "Senior Data Analyst",\n      "family": "analytics",\n      "onet_code": "15-2051.00",\n      "seniority": "senior",\n      "vector": {\n        "sql": 92, "data_modeling": 65, "bi_reporting": 90, "data_viz": 85, "statistics": 78,\n        "experimentation": 60, "python": 62, "software_eng": 30, "git_cicd": 35, "cloud": 40,\n        "data_engineering": 40, "dbt_semantic": 40, "machine_learning": 35, "mlops": 15,\n        "genai_llm": 30, "rag": 15, "ai_agents": 12, "ai_evaluation": 15, "prompt_eng": 35,\n        "business_analysis": 85, "stakeholder_mgmt": 85, "problem_framing": 80, "product_mgmt": 45,\n        "domain_knowledge": 80\n      },\n      "demand": 80, "growth": 58, "remote_share": 63, "ai_resilience": 58, "salary_index": 65\n    },\n    {\n      "id": "ai_analytics_lead",\n      "title": "AI-Augmented Analytics Lead",\n      "family": "analytics_ai",\n      "onet_code": "15-2051.00",\n      "seniority": "lead",\n      "vector": {\n        "sql": 85, "data_modeling": 70, "bi_reporting": 80, "data_viz": 78, "statistics": 75,\n        "experimentation": 65, "python": 75, "software_eng": 45, "git_cicd": 50, "cloud": 60,\n        "data_engineering": 55, "dbt_semantic": 60, "machine_learning": 55, "mlops": 40,\n        "genai_llm": 80, "rag": 70, "ai_agents": 65, "ai_evaluation": 70, "prompt_eng": 75,\n        "business_analysis": 85, "stakeholder_mgmt": 85, "problem_framing": 85, "product_mgmt": 60,\n        "domain_knowledge": 80\n      },\n      "demand": 82, "growth": 90, "remote_share": 68, "ai_resilience": 88, "salary_index": 80\n    },\n    {\n      "id": "analytics_engineer",\n      "title": "Analytics Engineer",\n      "family": "engineering",\n      "onet_code": "15-2051.00",\n      "seniority": "mid",\n      "vector": {\n        "sql": 90, "data_modeling": 85, "bi_reporting": 70, "data_viz": 60, "statistics": 60,\n        "experimentation": 50, "python": 75, "software_eng": 65, "git_cicd": 80, "cloud": 70,\n        "data_engineering": 75, "dbt_semantic": 90, "machine_learning": 40, "mlops": 45,\n        "genai_llm": 45, "rag": 35, "ai_agents": 25, "ai_evaluation": 35, "prompt_eng": 45,\n        "business_analysis": 65, "stakeholder_mgmt": 60, "problem_framing": 70, "product_mgmt": 40,\n        "domain_knowledge": 60\n      },\n      "demand": 88, "growth": 82, "remote_share": 72, "ai_resilience": 80, "salary_index": 78\n    },\n    {\n      "id": "ai_data_engineer",\n      "title": "AI Data Engineer",\n      "family": "engineering",\n      "onet_code": "15-1243.00",\n      "seniority": "senior",\n      "vector": {\n        "sql": 85, "data_modeling": 85, "bi_reporting": 45, "data_viz": 40, "statistics": 55,\n        "experimentation": 40, "python": 85, "software_eng": 80, "git_cicd": 85, "cloud": 85,\n        "data_engineering": 90, "dbt_semantic": 80, "machine_learning": 55, "mlops": 70,\n        "genai_llm": 70, "rag": 75, "ai_agents": 55, "ai_evaluation": 60, "prompt_eng": 60,\n        "business_analysis": 50, "stakeholder_mgmt": 55, "problem_framing": 65, "product_mgmt": 40,\n        "domain_knowledge": 55\n      },\n      "demand": 90, "growth": 88, "remote_share": 74, "ai_resilience": 85, "salary_index": 85\n    },\n    {\n      "id": "ai_analyst",\n      "title": "AI Analyst",\n      "family": "analytics_ai",\n      "onet_code": "15-2051.00",\n      "seniority": "mid",\n      "vector": {\n        "sql": 80, "data_modeling": 55, "bi_reporting": 70, "data_viz": 70, "statistics": 70,\n        "experimentation": 60, "python": 65, "software_eng": 35, "git_cicd": 40, "cloud": 50,\n        "data_engineering": 40, "dbt_semantic": 45, "machine_learning": 50, "mlops": 30,\n        "genai_llm": 75, "rag": 60, "ai_agents": 55, "ai_evaluation": 65, "prompt_eng": 75,\n        "business_analysis": 75, "stakeholder_mgmt": 70, "problem_framing": 75, "product_mgmt": 45,\n        "domain_knowledge": 70\n      },\n      "demand": 80, "growth": 85, "remote_share": 66, "ai_resilience": 85, "salary_index": 68\n    },\n    {\n      "id": "bi_lead",\n      "title": "BI Lead",\n      "family": "analytics",\n      "onet_code": "15-2051.00",\n      "seniority": "lead",\n      "vector": {\n        "sql": 88, "data_modeling": 75, "bi_reporting": 92, "data_viz": 88, "statistics": 65,\n        "experimentation": 55, "python": 55, "software_eng": 35, "git_cicd": 45, "cloud": 55,\n        "data_engineering": 50, "dbt_semantic": 65, "machine_learning": 30, "mlops": 20,\n        "genai_llm": 40, "rag": 25, "ai_agents": 20, "ai_evaluation": 30, "prompt_eng": 40,\n        "business_analysis": 85, "stakeholder_mgmt": 88, "problem_framing": 80, "product_mgmt": 60,\n        "domain_knowledge": 80\n      },\n      "demand": 74, "growth": 60, "remote_share": 60, "ai_resilience": 65, "salary_index": 75\n    },\n    {\n      "id": "analytics_manager",\n      "title": "Analytics Manager",\n      "family": "management",\n      "onet_code": "11-3021.00",\n      "seniority": "manager",\n      "vector": {\n        "sql": 70, "data_modeling": 60, "bi_reporting": 75, "data_viz": 70, "statistics": 65,\n        "experimentation": 60, "python": 45, "software_eng": 30, "git_cicd": 30, "cloud": 45,\n        "data_engineering": 40, "dbt_semantic": 45, "machine_learning": 35, "mlops": 20,\n        "genai_llm": 45, "rag": 25, "ai_agents": 25, "ai_evaluation": 40, "prompt_eng": 40,\n        "business_analysis": 88, "stakeholder_mgmt": 92, "problem_framing": 85, "product_mgmt": 70,\n        "domain_knowledge": 85\n      },\n      "demand": 72, "growth": 62, "remote_share": 55, "ai_resilience": 78, "salary_index": 82\n    },\n    {\n      "id": "ai_product_manager",\n      "title": "AI Data Product Manager",\n      "family": "product",\n      "onet_code": "11-2021.00",\n      "seniority": "senior",\n      "vector": {\n        "sql": 55, "data_modeling": 45, "bi_reporting": 60, "data_viz": 55, "statistics": 55,\n        "experimentation": 65, "python": 40, "software_eng": 35, "git_cicd": 30, "cloud": 45,\n        "data_engineering": 35, "dbt_semantic": 35, "machine_learning": 50, "mlops": 35,\n        "genai_llm": 70, "rag": 55, "ai_agents": 55, "ai_evaluation": 70, "prompt_eng": 60,\n        "business_analysis": 85, "stakeholder_mgmt": 90, "problem_framing": 88, "product_mgmt": 90,\n        "domain_knowledge": 80\n      },\n      "demand": 85, "growth": 90, "remote_share": 70, "ai_resilience": 90, "salary_index": 88\n    },\n    {\n      "id": "data_scientist",\n      "title": "Data Scientist",\n      "family": "science",\n      "onet_code": "15-2051.00",\n      "seniority": "senior",\n      "vector": {\n        "sql": 75, "data_modeling": 65, "bi_reporting": 55, "data_viz": 65, "statistics": 90,\n        "experimentation": 85, "python": 85, "software_eng": 55, "git_cicd": 60, "cloud": 60,\n        "data_engineering": 55, "dbt_semantic": 45, "machine_learning": 85, "mlops": 55,\n        "genai_llm": 65, "rag": 55, "ai_agents": 45, "ai_evaluation": 65, "prompt_eng": 55,\n        "business_analysis": 70, "stakeholder_mgmt": 65, "problem_framing": 80, "product_mgmt": 45,\n        "domain_knowledge": 65\n      },\n      "demand": 82, "growth": 78, "remote_share": 68, "ai_resilience": 80, "salary_index": 82\n    },\n    {\n      "id": "ml_engineer",\n      "title": "ML Engineer",\n      "family": "engineering",\n      "onet_code": "15-1252.00",\n      "seniority": "senior",\n      "vector": {\n        "sql": 65, "data_modeling": 65, "bi_reporting": 30, "data_viz": 40, "statistics": 80,\n        "experimentation": 60, "python": 90, "software_eng": 90, "git_cicd": 90, "cloud": 88,\n        "data_engineering": 80, "dbt_semantic": 50, "machine_learning": 92, "mlops": 88,\n        "genai_llm": 70, "rag": 65, "ai_agents": 60, "ai_evaluation": 70, "prompt_eng": 55,\n        "business_analysis": 45, "stakeholder_mgmt": 45, "problem_framing": 65, "product_mgmt": 35,\n        "domain_knowledge": 50\n      },\n      "demand": 92, "growth": 90, "remote_share": 72, "ai_resilience": 82, "salary_index": 92\n    },\n    {\n      "id": "ai_engineer",\n      "title": "AI Engineer (GenAI)",\n      "family": "engineering",\n      "onet_code": "15-1252.00",\n      "seniority": "senior",\n      "vector": {\n        "sql": 60, "data_modeling": 55, "bi_reporting": 30, "data_viz": 40, "statistics": 65,\n        "experimentation": 60, "python": 88, "software_eng": 85, "git_cicd": 85, "cloud": 82,\n        "data_engineering": 70, "dbt_semantic": 45, "machine_learning": 70, "mlops": 75,\n        "genai_llm": 92, "rag": 90, "ai_agents": 88, "ai_evaluation": 85, "prompt_eng": 88,\n        "business_analysis": 50, "stakeholder_mgmt": 50, "problem_framing": 70, "product_mgmt": 45,\n        "domain_knowledge": 55\n      },\n      "demand": 93, "growth": 95, "remote_share": 73, "ai_resilience": 90, "salary_index": 90\n    },\n    {\n      "id": "data_engineer",\n      "title": "Data Engineer",\n      "family": "engineering",\n      "onet_code": "15-1243.00",\n      "seniority": "senior",\n      "vector": {\n        "sql": 85, "data_modeling": 85, "bi_reporting": 40, "data_viz": 40, "statistics": 50,\n        "experimentation": 40, "python": 82, "software_eng": 80, "git_cicd": 85, "cloud": 85,\n        "data_engineering": 92, "dbt_semantic": 78, "machine_learning": 40, "mlops": 60,\n        "genai_llm": 45, "rag": 45, "ai_agents": 35, "ai_evaluation": 40, "prompt_eng": 40,\n        "business_analysis": 50, "stakeholder_mgmt": 50, "problem_framing": 60, "product_mgmt": 35,\n        "domain_knowledge": 55\n      },\n      "demand": 88, "growth": 80, "remote_share": 72, "ai_resilience": 75, "salary_index": 82\n    },\n    {\n      "id": "ai_solutions_architect",\n      "title": "AI Solutions Architect",\n      "family": "architecture",\n      "onet_code": "15-1299.08",\n      "seniority": "lead",\n      "vector": {\n        "sql": 65, "data_modeling": 70, "bi_reporting": 45, "data_viz": 45, "statistics": 60,\n        "experimentation": 55, "python": 75, "software_eng": 80, "git_cicd": 75, "cloud": 88,\n        "data_engineering": 75, "dbt_semantic": 60, "machine_learning": 65, "mlops": 75,\n        "genai_llm": 85, "rag": 82, "ai_agents": 80, "ai_evaluation": 80, "prompt_eng": 78,\n        "business_analysis": 75, "stakeholder_mgmt": 82, "problem_framing": 85, "product_mgmt": 65,\n        "domain_knowledge": 70\n      },\n      "demand": 86, "growth": 92, "remote_share": 70, "ai_resilience": 90, "salary_index": 90\n    },\n    {\n      "id": "ml_ops_engineer",\n      "title": "MLOps Engineer",\n      "family": "engineering",\n      "onet_code": "15-1252.00",\n      "seniority": "senior",\n      "vector": {\n        "sql": 55, "data_modeling": 55, "bi_reporting": 25, "data_viz": 30, "statistics": 55,\n        "experimentation": 45, "python": 85, "software_eng": 85, "git_cicd": 92, "cloud": 90,\n        "data_engineering": 75, "dbt_semantic": 45, "machine_learning": 70, "mlops": 92,\n        "genai_llm": 60, "rag": 55, "ai_agents": 50, "ai_evaluation": 70, "prompt_eng": 45,\n        "business_analysis": 40, "stakeholder_mgmt": 45, "problem_framing": 60, "product_mgmt": 30,\n        "domain_knowledge": 45\n      },\n      "demand": 85, "growth": 85, "remote_share": 74, "ai_resilience": 82, "salary_index": 86\n    }\n  ]\n}\n',
    'data/ai_task_impact.json': '{\n  "_note": "Task archetypes used to (1) build the current-role AI transformation table and (2) classify free-text job descriptions. \'impact\' is one of automated | ai_assisted | ai_augmented | human_critical. \'exposure\' is 0-100 (share of the task AI can perform). \'keywords\' are lowercase substrings used to match JD sentences to this archetype. \'human_value\' explains what stays human.",\n  "impact_levels": {\n    "automated":     { "label": "Automated",     "color": "red",    "exposure_default": 85, "meaning": "AI can perform most of this task with light oversight." },\n    "ai_assisted":   { "label": "AI-Assisted",   "color": "orange", "exposure_default": 65, "meaning": "AI drafts or accelerates; a human reviews and finalizes." },\n    "ai_augmented":  { "label": "AI-Augmented",  "color": "yellow", "exposure_default": 40, "meaning": "AI boosts a human who stays firmly in control." },\n    "human_critical":{ "label": "Human-Critical","color": "green",  "exposure_default": 12, "meaning": "Depends on judgement, trust, or accountability AI cannot own." }\n  },\n  "tasks": [\n    {\n      "id": "recurring_reporting",\n      "label": "Recurring reporting",\n      "impact": "automated",\n      "exposure": 85,\n      "keywords": ["recurring report", "weekly report", "monthly report", "scheduled report", "generate reports", "reporting", "kpi report", "status report"],\n      "human_value": "Deciding which metrics matter and how to act on them."\n    },\n    {\n      "id": "dashboard_creation",\n      "label": "Dashboard creation",\n      "impact": "ai_assisted",\n      "exposure": 70,\n      "keywords": ["dashboard", "power bi", "tableau", "looker", "visualiz", "reporting suite", "build dashboards"],\n      "human_value": "Choosing the right questions and designing for the audience."\n    },\n    {\n      "id": "sql_generation",\n      "label": "SQL / query generation",\n      "impact": "ai_assisted",\n      "exposure": 75,\n      "keywords": ["sql", "write queries", "query", "stored procedure", "data extraction", "pull data"],\n      "human_value": "Validating correctness against messy real-world data."\n    },\n    {\n      "id": "data_cleaning",\n      "label": "Data cleaning / wrangling",\n      "impact": "ai_assisted",\n      "exposure": 68,\n      "keywords": ["data cleaning", "data wrangling", "data preparation", "etl", "transform data", "clean data", "data quality"],\n      "human_value": "Judging which anomalies are errors vs. real signal."\n    },\n    {\n      "id": "exploratory_analysis",\n      "label": "Exploratory analysis",\n      "impact": "ai_augmented",\n      "exposure": 45,\n      "keywords": ["exploratory", "ad hoc analysis", "analyze data", "deep dive", "investigate", "root cause"],\n      "human_value": "Forming hypotheses and knowing what is worth exploring."\n    },\n    {\n      "id": "coding_implementation",\n      "label": "Coding / implementation",\n      "impact": "ai_assisted",\n      "exposure": 65,\n      "keywords": ["python", "develop", "implement", "build pipeline", "write code", "programming", "software", "scripting"],\n      "human_value": "Architecture decisions and reviewing AI-generated code."\n    },\n    {\n      "id": "documentation",\n      "label": "Documentation",\n      "impact": "automated",\n      "exposure": 80,\n      "keywords": ["documentation", "document", "write docs", "technical writing", "knowledge base"],\n      "human_value": "Confirming accuracy and capturing tacit knowledge."\n    },\n    {\n      "id": "model_building",\n      "label": "Model building / ML",\n      "impact": "ai_augmented",\n      "exposure": 50,\n      "keywords": ["machine learning", "model", "predictive", "train model", "ml", "forecasting", "algorithm"],\n      "human_value": "Framing the problem and validating that the model is trustworthy."\n    },\n    {\n      "id": "business_interpretation",\n      "label": "Business interpretation of results",\n      "impact": "human_critical",\n      "exposure": 18,\n      "keywords": ["insight", "interpret", "recommend", "business impact", "actionable", "translate data", "storytelling", "narrative"],\n      "human_value": "Connecting numbers to decisions and owning the call."\n    },\n    {\n      "id": "stakeholder_management",\n      "label": "Stakeholder management",\n      "impact": "human_critical",\n      "exposure": 12,\n      "keywords": ["stakeholder", "collaborate", "cross-functional", "partner with", "communicate", "present to", "influence", "align"],\n      "human_value": "Building trust and navigating competing priorities."\n    },\n    {\n      "id": "problem_framing",\n      "label": "Problem framing / scoping",\n      "impact": "human_critical",\n      "exposure": 15,\n      "keywords": ["define requirements", "scope", "problem framing", "identify opportunities", "strategy", "roadmap", "prioritize"],\n      "human_value": "Deciding which problem is worth solving in the first place."\n    },\n    {\n      "id": "leadership_mentoring",\n      "label": "Leadership / mentoring",\n      "impact": "human_critical",\n      "exposure": 10,\n      "keywords": ["mentor", "lead", "manage team", "coach", "hire", "people management", "guide", "supervise"],\n      "human_value": "Developing people and setting direction."\n    },\n    {\n      "id": "governance_compliance",\n      "label": "Governance / compliance judgement",\n      "impact": "human_critical",\n      "exposure": 22,\n      "keywords": ["governance", "compliance", "regulatory", "privacy", "ethics", "risk", "audit", "policy"],\n      "human_value": "Accountable judgement calls regulators expect from a person."\n    },\n    {\n      "id": "data_pipeline_ops",\n      "label": "Pipeline / infra operations",\n      "impact": "ai_assisted",\n      "exposure": 60,\n      "keywords": ["pipeline", "orchestration", "airflow", "infrastructure", "deploy", "ci/cd", "monitoring", "cloud"],\n      "human_value": "Designing for reliability and handling novel failures."\n    },\n    {\n      "id": "ai_evaluation_task",\n      "label": "AI system evaluation",\n      "impact": "ai_augmented",\n      "exposure": 40,\n      "keywords": ["evaluation", "eval", "guardrail", "hallucination", "model quality", "llm", "rag", "prompt"],\n      "human_value": "Defining what \'good\' means and judging edge cases."\n    }\n  ]\n}\n',
    'src/__init__.py': '"""Data Career Navigator - self-contained runtime package."""\n',
    'src/data_loader.py': '"""\ndata_loader.py\n--------------\nLoads the bundled open seed dataset (roles, skills, AI task impact).\n\nTo make the engine authoritative, replace the JSON files in ../data with\ndata derived from O*NET (tasks/skills), BLS (growth/demand) and an\nopen-licensed job-postings dataset (market demand / remote share), keeping\nthe same schema described below.\n\nSchemas\n-------\nskills.json   -> { "skills": [{id, label, category}], transferable_categories, technical_categories }\nroles.json    -> { "roles": [{id, title, family, onet_code, seniority, vector{skill_id:0-100},\n                              demand, growth, remote_share, ai_resilience, salary_index}] }\nai_task_impact.json -> { impact_levels{...}, "tasks": [{id, label, impact, exposure, keywords[], human_value}] }\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom functools import lru_cache\nfrom pathlib import Path\nfrom typing import Any\n\n# data/ lives next to src/ at the project root\nDATA_DIR = Path(__file__).resolve().parent.parent / "data"\n\n\ndef _read_json(name: str) -> dict[str, Any]:\n    path = DATA_DIR / name\n    with path.open("r", encoding="utf-8") as fh:\n        return json.load(fh)\n\n\n@lru_cache(maxsize=1)\ndef load_skills() -> dict[str, Any]:\n    """Canonical skill dimensions + category groupings."""\n    return _read_json("skills.json")\n\n\n@lru_cache(maxsize=1)\ndef load_roles() -> dict[str, Any]:\n    """Role skill vectors + market signals."""\n    return _read_json("roles.json")\n\n\n@lru_cache(maxsize=1)\ndef load_task_impact() -> dict[str, Any]:\n    """AI task archetypes for the transformation table and JD analyzer."""\n    return _read_json("ai_task_impact.json")\n\n\ndef skill_ids() -> list[str]:\n    """Ordered list of canonical skill ids (defines vector dimension order)."""\n    return [s["id"] for s in load_skills()["skills"]]\n\n\ndef skill_labels() -> dict[str, str]:\n    return {s["id"]: s["label"] for s in load_skills()["skills"]}\n\n\ndef skill_categories() -> dict[str, str]:\n    return {s["id"]: s["category"] for s in load_skills()["skills"]}\n\n\ndef transferable_skill_ids() -> set[str]:\n    cats = set(load_skills()["transferable_categories"])\n    return {s["id"] for s in load_skills()["skills"] if s["category"] in cats}\n\n\ndef roles_by_id() -> dict[str, dict[str, Any]]:\n    return {r["id"]: r for r in load_roles()["roles"]}\n\n\ndef get_role(role_id: str) -> dict[str, Any]:\n    roles = roles_by_id()\n    if role_id not in roles:\n        raise KeyError(f"Unknown role id \'{role_id}\'. Known: {sorted(roles)}")\n    return roles[role_id]\n',
    'src/engine.py': '"""\nengine.py\n---------\nDeterministic scoring engine for the Data Career Navigator.\n\nDesign principle: career advice is a *data problem*, not a black-box answer.\nEvery score is computed with transparent math over skill vectors and market\nsignals, and every result carries an `explanation` so the UI can answer\n"Why am I being recommended this role?".\n\nNo network calls, no API key required. Runs fully offline.\n\nMain entry point\n----------------\n    from src.engine import analyze_profile\n    result = analyze_profile(profile)\n\n`profile` is a dict:\n    {\n      "title": "Senior Data Analyst",\n      "years_experience": 9,\n      "skills": {"sql": 90, "bi_reporting": 85, "python": 55, ...},  # 0-100\n      "industry": "Banking",\n      "location": "India",\n      "remote_preference": True,\n      "interests": ["genai_llm", "business_analysis"],  # skill ids or free text\n    }\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Any\n\nimport numpy as np\n\nfrom . import data_loader as dl\n\n# ---------------------------------------------------------------------------\n# Vector helpers\n# ---------------------------------------------------------------------------\n\ndef _vector_from_map(skill_map: dict[str, float]) -> np.ndarray:\n    """Turn a {skill_id: value} map into an ordered numpy vector (0-100)."""\n    ids = dl.skill_ids()\n    return np.array([float(skill_map.get(sid, 0.0)) for sid in ids], dtype=float)\n\n\ndef role_vector(role: dict[str, Any]) -> np.ndarray:\n    return _vector_from_map(role.get("vector", {}))\n\n\ndef profile_vector(profile: dict[str, Any]) -> np.ndarray:\n    return _vector_from_map(profile.get("skills", {}))\n\n\ndef career_distance(vec_a: np.ndarray, vec_b: np.ndarray) -> float:\n    """\n    Symmetric Euclidean \'skill distance\' between two roles, normalized to a\n    0-100 point scale so it reads like "you are N skill points away". This is\n    the human-readable distance shown in the UI.\n    """\n    diff = vec_a - vec_b\n    rms = float(np.sqrt(np.mean(diff ** 2)))\n    return round(rms, 1)\n\n\ndef directional_shortfall(profile_vec: np.ndarray, target_vec: np.ndarray) -> float:\n    """\n    How far the profile falls SHORT of the target\'s requirements.\n\n    Only counts skills where the user is below what the target needs (having\n    extra, irrelevant skills is not a penalty). Averaged only over the skills\n    the target actually requires, and scaled by how demanding each requirement\n    is. Returns a 0-100 "points to close" figure.\n    """\n    required = target_vec > 0\n    if not np.any(required):\n        return 0.0\n    shortfall = np.clip(target_vec - profile_vec, 0, None)\n    # weight the shortfall by requirement strength: missing a core skill hurts\n    # more than missing a lightly-required one.\n    weights = target_vec[required]\n    gaps = shortfall[required]\n    weighted = float(np.sum(gaps * weights) / np.sum(weights))\n    return round(weighted, 1)\n\n\ndef cosine_similarity(vec_a: np.ndarray, vec_b: np.ndarray) -> float:\n    denom = np.linalg.norm(vec_a) * np.linalg.norm(vec_b)\n    if denom == 0:\n        return 0.0\n    return float(np.dot(vec_a, vec_b) / denom)\n\n\n# ---------------------------------------------------------------------------\n# 1. Current profile analysis\n# ---------------------------------------------------------------------------\n\ndef classify_skills(profile: dict[str, Any]) -> dict[str, list[dict[str, Any]]]:\n    """Bucket the user\'s skills into strong / moderate / emerging."""\n    labels = dl.skill_labels()\n    strong, moderate, emerging = [], [], []\n    for sid, val in profile.get("skills", {}).items():\n        entry = {"id": sid, "label": labels.get(sid, sid), "level": val}\n        if val >= 70:\n            strong.append(entry)\n        elif val >= 40:\n            moderate.append(entry)\n        else:\n            emerging.append(entry)\n    for bucket in (strong, moderate, emerging):\n        bucket.sort(key=lambda e: e["level"], reverse=True)\n    return {"strong": strong, "moderate": moderate, "emerging": emerging}\n\n\ndef role_transformation_score(profile: dict[str, Any]) -> dict[str, Any]:\n    """\n    AI transformation score for the user\'s CURRENT role.\n\n    We match the current role to the closest seed role, then aggregate the AI\n    exposure of the tasks that role emphasises, weighted by how present each\n    task\'s skills are in the profile. Framed as \'task transformation\', not risk\n    of job loss.\n    """\n    from .jd_analyzer import score_tasks_for_role  # local import avoids cycle\n\n    matched = closest_role(profile)\n    task_rows = score_tasks_for_role(matched["role"])\n    if task_rows:\n        weighted = sum(t["exposure"] * t["weight"] for t in task_rows)\n        total_w = sum(t["weight"] for t in task_rows)\n        score = round(weighted / total_w) if total_w else 0\n    else:\n        score = 0\n    return {\n        "score": score,\n        "matched_role": matched["role"]["title"],\n        "matched_role_id": matched["role"]["id"],\n        "similarity": round(matched["similarity"], 3),\n        "tasks": task_rows,\n        "framing": (\n            "Your role is undergoing significant task transformation. "\n            "This measures how much of the work is being reshaped by AI, "\n            "not the risk of your job disappearing."\n        ),\n    }\n\n\n# ---------------------------------------------------------------------------\n# 2. Role matching\n# ---------------------------------------------------------------------------\n\ndef closest_role(profile: dict[str, Any]) -> dict[str, Any]:\n    """Find the seed role whose vector is most similar to the profile."""\n    p_vec = profile_vector(profile)\n    best = None\n    for role in dl.load_roles()["roles"]:\n        sim = cosine_similarity(p_vec, role_vector(role))\n        if best is None or sim > best["similarity"]:\n            best = {"role": role, "similarity": sim}\n    return best\n\n\n# ---------------------------------------------------------------------------\n# 3. Transferability, gaps, and transition scoring\n# ---------------------------------------------------------------------------\n\ndef skill_gap(profile: dict[str, Any], role: dict[str, Any]) -> dict[str, Any]:\n    """\n    Compare profile vs a target role. Returns skills the user already has vs\n    those they need to build (target requires notably more than user has).\n    """\n    labels = dl.skill_labels()\n    have, need = [], []\n    for sid, required in role.get("vector", {}).items():\n        current = float(profile.get("skills", {}).get(sid, 0.0))\n        gap = required - current\n        entry = {\n            "id": sid,\n            "label": labels.get(sid, sid),\n            "required": required,\n            "current": current,\n            "gap": round(gap, 1),\n        }\n        if gap >= 20 and required >= 45:  # meaningfully short on a skill that matters\n            need.append(entry)\n        elif current >= max(40, required - 15):\n            have.append(entry)\n    need.sort(key=lambda e: e["gap"], reverse=True)\n    have.sort(key=lambda e: e["current"], reverse=True)\n    return {"have": have, "need": need}\n\n\ndef transferability(profile: dict[str, Any], role: dict[str, Any]) -> dict[str, Any]:\n    """\n    % of the target role\'s required skills already met by the profile,\n    with extra weight on transferable (business/analytics) skills that carry\n    across roles.\n    """\n    transfer_ids = dl.transferable_skill_ids()\n    num, den = 0.0, 0.0\n    for sid, required in role.get("vector", {}).items():\n        if required <= 0:\n            continue\n        current = float(profile.get("skills", {}).get(sid, 0.0))\n        coverage = min(current / required, 1.0)\n        weight = 1.5 if sid in transfer_ids else 1.0\n        num += coverage * required * weight\n        den += required * weight\n    pct = round((num / den) * 100) if den else 0\n    return {"transferable_pct": pct}\n\n\ndef experience_bonus(years: float) -> float:\n    """Small bonus for domain experience, capped. 0 yrs -> 0, 10+ yrs -> ~1.0."""\n    return min(years / 10.0, 1.0)\n\n\ndef transition_score(profile: dict[str, Any], role: dict[str, Any]) -> dict[str, Any]:\n    """\n    How realistic is moving from the profile to this role?\n    Blend skill fit, skill distance (closeness), and experience.\n    """\n    p_vec = profile_vector(profile)\n    r_vec = role_vector(role)\n    distance = career_distance(p_vec, r_vec)\n    shortfall = directional_shortfall(p_vec, r_vec)\n    transfer = transferability(profile, role)["transferable_pct"]\n    exp = experience_bonus(profile.get("years_experience", 0))\n\n    # readiness: how close the user already is to meeting the target\'s bar.\n    # shortfall is the "points to close"; readiness is its complement.\n    readiness = max(0.0, 100.0 - shortfall)\n\n    # weighted blend of transferable coverage, readiness, and experience\n    score = (\n        0.45 * transfer +\n        0.40 * readiness +\n        0.15 * (exp * 100)\n    )\n    score = round(min(max(score, 0), 100))\n\n    return {\n        "score": score,\n        "skill_distance": distance,\n        "skill_shortfall": shortfall,\n        "readiness": round(readiness, 1),\n        "transferable_pct": transfer,\n        "experience_factor": round(exp, 2),\n        "explanation": {\n            "formula": "0.45*transferable% + 0.40*(100 - skill_shortfall) + 0.15*(experience_factor*100)",\n            "transferable_pct": transfer,\n            "skill_shortfall": shortfall,\n            "readiness": round(readiness, 1),\n            "skill_distance": distance,\n            "experience_factor": round(exp, 2),\n        },\n    }\n\n\n# ---------------------------------------------------------------------------\n# 4. Future Fit + Remote Fit\n# ---------------------------------------------------------------------------\n\nFUTURE_FIT_WEIGHTS = {\n    "skill_fit": 0.30,      # transition score (transferability + closeness)\n    "demand": 0.20,         # current market demand\n    "growth": 0.20,         # projected growth\n    "ai_resilience": 0.20,  # augmented rather than automated\n    "salary": 0.10,         # relative salary potential\n}\n\n\ndef future_fit(profile: dict[str, Any], role: dict[str, Any],\n               transition: dict[str, Any]) -> dict[str, Any]:\n    """\n    Composite \'is this a good destination?\' score across five dimensions.\n    Remote fit is reported separately so the user can weigh it by preference.\n    """\n    components = {\n        "skill_fit": transition["score"],\n        "demand": role.get("demand", 0),\n        "growth": role.get("growth", 0),\n        "ai_resilience": role.get("ai_resilience", 0),\n        "salary": role.get("salary_index", 0),\n    }\n    overall = round(sum(components[k] * w for k, w in FUTURE_FIT_WEIGHTS.items()))\n    return {\n        "score": overall,\n        "components": components,\n        "weights": FUTURE_FIT_WEIGHTS,\n        "explanation": {\n            "formula": " + ".join(f"{w}*{k}" for k, w in FUTURE_FIT_WEIGHTS.items()),\n            "components": components,\n        },\n    }\n\n\ndef remote_fit(profile: dict[str, Any], role: dict[str, Any]) -> dict[str, Any]:\n    """Remote suitability of the target role vs the user\'s preference."""\n    share = role.get("remote_share", 0)\n    prefers_remote = bool(profile.get("remote_preference", False))\n    note = (\n        "Strong remote availability." if share >= 70 else\n        "Moderate remote availability." if share >= 55 else\n        "Limited remote availability."\n    )\n    if prefers_remote and share < 55:\n        note += " May conflict with your remote preference."\n    return {"score": share, "prefers_remote": prefers_remote, "note": note}\n\n\n# ---------------------------------------------------------------------------\n# 5. Roadmap + portfolio projects\n# ---------------------------------------------------------------------------\n\ndef _skill_learning_action(sid: str) -> str:\n    actions = {\n        "python": "Python for data work (pandas, typing, testing)",\n        "git_cicd": "Git workflows + basic CI/CD",\n        "dbt_semantic": "dbt models + a semantic/metrics layer",\n        "data_engineering": "Batch & streaming ETL fundamentals",\n        "cloud": "One cloud platform (storage, compute, IAM basics)",\n        "genai_llm": "LLM fundamentals and the major model APIs",\n        "rag": "Retrieval-augmented generation + vector search",\n        "ai_agents": "Agentic patterns (tools, planning, orchestration)",\n        "ai_evaluation": "AI evaluation: metrics, guardrails, red-teaming",\n        "prompt_eng": "Structured prompting and prompt evaluation",\n        "machine_learning": "Core ML modelling and validation",\n        "mlops": "Model deployment, monitoring, and MLOps",\n        "software_eng": "Software engineering practices (design, review)",\n        "product_mgmt": "Product discovery and lifecycle for AI products",\n        "data_modeling": "Dimensional + analytical data modeling",\n    }\n    return actions.get(sid, dl.skill_labels().get(sid, sid))\n\n\ndef build_roadmap(gap: dict[str, Any]) -> dict[str, Any]:\n    """Split the top skill gaps into a 3 x 30-day plan."""\n    needed = [g["id"] for g in gap["need"]]\n    # order by category so foundational/engineering skills come first\n    cats = dl.skill_categories()\n    order = {"engineering": 0, "data": 1, "ai": 2, "analytics": 3, "business": 4}\n    needed.sort(key=lambda s: order.get(cats.get(s, ""), 5))\n\n    thirds = [needed[i::3] for i in range(3)] if needed else [[], [], []]\n    # distribute more sensibly: first phase = foundations\n    n = len(needed)\n    p1 = needed[: max(1, n // 3)] if n else []\n    p2 = needed[len(p1): len(p1) + max(1, n // 3)] if n else []\n    p3 = needed[len(p1) + len(p2):] if n else []\n\n    def phase(ids: list[str]) -> list[str]:\n        return [_skill_learning_action(s) for s in ids]\n\n    return {\n        "days_1_30": {"theme": "Foundations", "focus": phase(p1) or ["Reinforce existing strengths"]},\n        "days_31_60": {"theme": "Core new skills", "focus": phase(p2) or ["Deepen applied practice"]},\n        "days_61_90": {"theme": "Build & evaluate", "focus": phase(p3) or ["Ship a capstone project"]},\n    }\n\n\nPORTFOLIO_PROJECTS = {\n    "analytics_ai": "Build an AI analytics agent that turns business questions into validated SQL and an executive insight summary.",\n    "engineering": "Build an end-to-end dbt + orchestration pipeline with tests, docs, and CI.",\n    "product": "Write an AI product spec + build a working prototype with an evaluation harness.",\n    "management": "Design an analytics operating model and a metrics governance framework with a demo.",\n    "science": "Build a forecasting/ML project with a clear validation and monitoring story.",\n    "architecture": "Design and prototype a RAG reference architecture with evaluation and guardrails.",\n    "analytics": "Build a self-serve BI + semantic layer with an AI question-answering front end.",\n}\n\n\ndef portfolio_project(role: dict[str, Any]) -> str:\n    return PORTFOLIO_PROJECTS.get(\n        role.get("family", ""),\n        "Build a hands-on project that demonstrates the target role\'s core skills end to end.",\n    )\n\n\ndef estimate_transition_time(transition_score_val: int) -> str:\n    if transition_score_val >= 85:\n        return "3-6 months"\n    if transition_score_val >= 75:\n        return "4-8 months"\n    if transition_score_val >= 60:\n        return "6-12 months"\n    return "9-18 months"\n\n\n# ---------------------------------------------------------------------------\n# 6. Top-level orchestration\n# ---------------------------------------------------------------------------\n\ndef recommend_transitions(profile: dict[str, Any], top_n: int = 5) -> list[dict[str, Any]]:\n    """Score every seed role as a possible destination and rank them."""\n    current = closest_role(profile)\n    current_id = current["role"]["id"]\n\n    results = []\n    for role in dl.load_roles()["roles"]:\n        if role["id"] == current_id:\n            continue  # don\'t recommend the role they\'re already in\n        trans = transition_score(profile, role)\n        fit = future_fit(profile, role, trans)\n        gap = skill_gap(profile, role)\n        rfit = remote_fit(profile, role)\n        results.append({\n            "role_id": role["id"],\n            "title": role["title"],\n            "family": role.get("family"),\n            "onet_code": role.get("onet_code"),\n            "transition_score": trans["score"],\n            "future_fit": fit["score"],\n            "remote_fit": rfit["score"],\n            "skill_distance": trans["skill_distance"],\n            "transferable_pct": trans["transferable_pct"],\n            "estimated_time": estimate_transition_time(trans["score"]),\n            "skill_gap": gap,\n            "roadmap": build_roadmap(gap),\n            "portfolio_project": portfolio_project(role),\n            "remote_note": rfit["note"],\n            "future_fit_detail": fit,\n            "transition_detail": trans,\n            "market": {\n                "demand": role.get("demand"),\n                "growth": role.get("growth"),\n                "remote_share": role.get("remote_share"),\n                "ai_resilience": role.get("ai_resilience"),\n                "salary_index": role.get("salary_index"),\n            },\n        })\n\n    # rank primarily by future fit, but blend with transition realism\n    results.sort(\n        key=lambda r: 0.6 * r["future_fit"] + 0.4 * r["transition_score"],\n        reverse=True,\n    )\n    return results[:top_n]\n\n\ndef analyze_profile(profile: dict[str, Any], top_n: int = 5) -> dict[str, Any]:\n    """Full Career Navigator analysis for a user profile."""\n    return {\n        "profile": {\n            "title": profile.get("title"),\n            "years_experience": profile.get("years_experience"),\n            "industry": profile.get("industry"),\n            "location": profile.get("location"),\n            "remote_preference": profile.get("remote_preference"),\n        },\n        "skill_profile": classify_skills(profile),\n        "current_role_match": closest_role(profile)["role"]["title"],\n        "ai_transformation": role_transformation_score(profile),\n        "recommendations": recommend_transitions(profile, top_n=top_n),\n    }\n',
    'src/jd_analyzer.py': '"""\njd_analyzer.py\n--------------\nJob Description AI-Exposure Analyzer.\n\nTwo entry points:\n\n1. analyze_jd(text) -> classify a free-text job description into:\n     - overall "AI takeover %"\n     - actions AI can take over\n     - actions that remain human-critical\n     - human + AI hybrid tasks\n     - a task-level breakdown table with rationale\n\n2. score_tasks_for_role(role) -> used by engine.role_transformation_score to\n   build the current-role AI transformation table from a role\'s skill vector.\n\nDeterministic and offline: JD parsing uses sentence splitting + keyword\nmatching against the task archetypes in data/ai_task_impact.json. An optional\nLLM layer can be plugged in later for richer parsing, but is never required.\n"""\n\nfrom __future__ import annotations\n\nimport re\nfrom typing import Any\n\nfrom . import data_loader as dl\n\n# ---------------------------------------------------------------------------\n# Task archetype access\n# ---------------------------------------------------------------------------\n\ndef _tasks() -> list[dict[str, Any]]:\n    return dl.load_task_impact()["tasks"]\n\n\ndef _impact_levels() -> dict[str, Any]:\n    return dl.load_task_impact()["impact_levels"]\n\n\n# skill_id -> which task archetypes that skill drives (for role-based table)\n_SKILL_TO_TASKS: dict[str, list[str]] = {\n    "sql": ["sql_generation", "data_cleaning"],\n    "data_modeling": ["coding_implementation", "data_pipeline_ops"],\n    "bi_reporting": ["recurring_reporting", "dashboard_creation"],\n    "data_viz": ["dashboard_creation"],\n    "statistics": ["exploratory_analysis", "model_building"],\n    "experimentation": ["exploratory_analysis", "model_building"],\n    "python": ["coding_implementation", "data_cleaning"],\n    "software_eng": ["coding_implementation", "data_pipeline_ops"],\n    "git_cicd": ["data_pipeline_ops"],\n    "cloud": ["data_pipeline_ops"],\n    "data_engineering": ["data_pipeline_ops", "data_cleaning"],\n    "dbt_semantic": ["coding_implementation", "data_pipeline_ops"],\n    "machine_learning": ["model_building"],\n    "mlops": ["data_pipeline_ops", "model_building"],\n    "genai_llm": ["ai_evaluation_task", "coding_implementation"],\n    "rag": ["ai_evaluation_task"],\n    "ai_agents": ["ai_evaluation_task"],\n    "ai_evaluation": ["ai_evaluation_task"],\n    "prompt_eng": ["ai_evaluation_task"],\n    "business_analysis": ["business_interpretation", "problem_framing"],\n    "stakeholder_mgmt": ["stakeholder_management"],\n    "problem_framing": ["problem_framing"],\n    "product_mgmt": ["problem_framing", "stakeholder_management"],\n    "domain_knowledge": ["business_interpretation"],\n}\n\n\n# ---------------------------------------------------------------------------\n# Role-based transformation table (used by engine)\n# ---------------------------------------------------------------------------\n\ndef score_tasks_for_role(role: dict[str, Any]) -> list[dict[str, Any]]:\n    """\n    Build a task-level AI-impact table for a role from its skill vector.\n\n    A task\'s weight = how strongly the role\'s skills drive that task. We only\n    include tasks whose driving skills are meaningfully present in the role.\n    """\n    tasks_by_id = {t["id"]: t for t in _tasks()}\n    levels = _impact_levels()\n    vector = role.get("vector", {})\n\n    weights: dict[str, float] = {}\n    for sid, val in vector.items():\n        for task_id in _SKILL_TO_TASKS.get(sid, []):\n            weights[task_id] = weights.get(task_id, 0.0) + float(val)\n\n    rows = []\n    for task_id, weight in weights.items():\n        if weight < 40:  # skill signal too weak to say the role does this task\n            continue\n        task = tasks_by_id[task_id]\n        level = task["impact"]\n        rows.append({\n            "task_id": task_id,\n            "task": task["label"],\n            "impact": level,\n            "impact_label": levels[level]["label"],\n            "color": levels[level]["color"],\n            "exposure": task["exposure"],\n            "weight": round(weight, 1),\n            "human_value": task["human_value"],\n            "what_happens": levels[level]["meaning"],\n        })\n    rows.sort(key=lambda r: r["exposure"], reverse=True)\n    return rows\n\n\n# ---------------------------------------------------------------------------\n# Free-text JD parsing\n# ---------------------------------------------------------------------------\n\ndef _split_sentences(text: str) -> list[str]:\n    """Split a JD into candidate task lines: by newline, bullet, and sentence."""\n    # normalize bullets and semicolons into line breaks\n    text = re.sub(r"[•\\u2022\\-\\*]\\s+", "\\n", text)\n    parts = re.split(r"[\\n\\r]+|(?<=[.;])\\s+", text)\n    lines = [p.strip() for p in parts if len(p.strip()) >= 12]\n    return lines\n\n\ndef _match_tasks_in_line(line: str) -> list[dict[str, Any]]:\n    """Return every task archetype whose keywords appear in the line."""\n    low = line.lower()\n    matches = []\n    for task in _tasks():\n        for kw in task["keywords"]:\n            if kw in low:\n                matches.append(task)\n                break\n    return matches\n\n\ndef analyze_jd(text: str) -> dict[str, Any]:\n    """\n    Analyze a free-text job description.\n\n    Returns overall AI takeover %, categorized action lists, and a task-level\n    breakdown with rationale. Every number is explainable.\n    """\n    levels = _impact_levels()\n    lines = _split_sentences(text)\n\n    # dedupe matched tasks but keep an example line for each\n    matched: dict[str, dict[str, Any]] = {}\n    for line in lines:\n        for task in _match_tasks_in_line(line):\n            if task["id"] not in matched:\n                matched[task["id"]] = {"task": task, "example": line}\n\n    breakdown = []\n    for entry in matched.values():\n        task = entry["task"]\n        level = task["impact"]\n        breakdown.append({\n            "task_id": task["id"],\n            "task": task["label"],\n            "impact": level,\n            "impact_label": levels[level]["label"],\n            "color": levels[level]["color"],\n            "exposure": task["exposure"],\n            "what_happens": levels[level]["meaning"],\n            "human_value": task["human_value"],\n            "matched_text": entry["example"],\n        })\n    breakdown.sort(key=lambda r: r["exposure"], reverse=True)\n\n    if breakdown:\n        takeover = round(sum(r["exposure"] for r in breakdown) / len(breakdown))\n    else:\n        takeover = 0\n\n    ai_takeover_actions = [\n        r for r in breakdown if r["impact"] in ("automated", "ai_assisted")\n    ]\n    hybrid_actions = [r for r in breakdown if r["impact"] == "ai_augmented"]\n    human_actions = [r for r in breakdown if r["impact"] == "human_critical"]\n\n    return {\n        "ai_takeover_pct": takeover,\n        "summary": _summary(takeover, len(breakdown)),\n        "tasks_detected": len(breakdown),\n        "ai_takeover_actions": ai_takeover_actions,\n        "hybrid_actions": hybrid_actions,\n        "human_critical_actions": human_actions,\n        "breakdown": breakdown,\n        "explanation": {\n            "method": (\n                "The JD is split into task lines and matched against known task "\n                "archetypes by keyword. The AI takeover % is the average AI "\n                "exposure across the detected tasks. Framed as task transformation, "\n                "not job elimination."\n            ),\n            "formula": "mean(exposure of detected tasks)",\n        },\n    }\n\n\ndef _summary(takeover: int, n_tasks: int) -> str:\n    if n_tasks == 0:\n        return ("No recognizable tasks were detected. Try pasting the "\n                "responsibilities / requirements section of the JD.")\n    if takeover >= 70:\n        band = "highly exposed to AI transformation"\n    elif takeover >= 50:\n        band = "substantially reshaped by AI, with humans reviewing and directing"\n    elif takeover >= 35:\n        band = "AI-augmented, with humans firmly in control"\n    else:\n        band = "anchored in human judgement, with AI in a supporting role"\n    return (f"About {takeover}% of this role\'s detected tasks are exposed to AI. "\n            f"The role is {band}.")\n',
}
for _path, _content in _FILES.items():
    with open(_path, 'w', encoding='utf-8') as _fh:
        _fh.write(_content)

# 3) Make src/ importable and load the engine.
sys.path.insert(0, os.getcwd())
import importlib
import src.data_loader, src.engine, src.jd_analyzer
importlib.reload(src.data_loader); importlib.reload(src.jd_analyzer); importlib.reload(src.engine)

import pandas as pd
from src.engine import analyze_profile, recommend_transitions
from src.jd_analyzer import analyze_jd
print('Setup OK - engine + data written to runtime, running fully offline.')


## 1. Describe your profile

Skills are rated **0-100** (your self-assessed proficiency). Anything you leave out is treated as 0 (emerging).

Canonical skill ids: `sql, data_modeling, bi_reporting, data_viz, statistics, experimentation, python, software_eng, git_cicd, cloud, data_engineering, dbt_semantic, machine_learning, mlops, genai_llm, rag, ai_agents, ai_evaluation, prompt_eng, business_analysis, stakeholder_mgmt, problem_framing, product_mgmt, domain_knowledge`

In [ ]:
profile = {
    'title': 'Senior Data Analyst',
    'years_experience': 9,
    'skills': {
        'sql': 90, 'bi_reporting': 85, 'data_viz': 80, 'statistics': 70,
        'python': 55, 'data_modeling': 55,
        'business_analysis': 80, 'stakeholder_mgmt': 80, 'problem_framing': 70,
        'domain_knowledge': 75,
        'genai_llm': 25,
    },
    'industry': 'Banking',
    'location': 'India',
    'remote_preference': True,
    'interests': ['genai_llm', 'business_analysis'],
}

result = analyze_profile(profile, top_n=5)
print('Closest current role match:', result['current_role_match'])

## 2. Where you are today

Your skills bucketed into **strong / moderate / emerging**.

In [ ]:
sp = result['skill_profile']
for bucket in ('strong', 'moderate', 'emerging'):
    items = ', '.join(f"{e['label']} ({e['level']})" for e in sp[bucket])
    print(f"{bucket.upper():9}: {items if items else '-'}")

## 3. How AI is transforming your current role

A **task transformation** score, not a job-loss score. It shows which parts of the work AI reshapes, and which parts stay human.

In [ ]:
ai = result['ai_transformation']
print(f"Current role transformation score: {ai['score']}/100")
print(ai['framing'])
print()

impact_icon = {'automated': 'HIGH (automated)', 'ai_assisted': 'HIGH (AI-assisted)',
               'ai_augmented': 'MEDIUM (AI-augmented)', 'human_critical': 'LOW (stays human)'}

df_tasks = pd.DataFrame([
    {'Task': t['task'], 'AI impact': impact_icon[t['impact']],
     'Exposure': t['exposure'], 'What happens': t['what_happens']}
    for t in ai['tasks']
])
df_tasks

## 4. Your top recommended next roles

Each destination gets a **Transition Score** (how realistic the move is) and a **Future Fit Score** (how good a destination it is). Roles are ranked by a blend of both.

In [ ]:
recs = result['recommendations']
df_recs = pd.DataFrame([
    {'Role': r['title'], 'Transition': r['transition_score'], 'Future Fit': r['future_fit'],
     'Remote Fit': r['remote_fit'], 'Skill shortfall': r['transition_detail']['skill_shortfall'],
     'Est. time': r['estimated_time']}
    for r in recs
])
df_recs

## 5. Deep dive on the top recommendation

Skill gap, remote fit, 90-day plan, and a portfolio project.

In [ ]:
top = recs[0]
print(f"=== {top['title']} ===")
print(f"Transition score: {top['transition_score']}/100 | Future fit: {top['future_fit']}/100 | "
      f"Remote fit: {top['remote_fit']}/100 | Est. time: {top['estimated_time']}")
print(f"O*NET occupation (for grounding): {top['onet_code']}")
print(f"Remote: {top['remote_note']}\n")

print('You already have:')
for h in top['skill_gap']['have'][:8]:
    print(f"  - {h['label']} ({int(h['current'])})")

print('\nYou need to build:')
for n in top['skill_gap']['need']:
    print(f"  - {n['label']}: have {int(n['current'])} -> need {int(n['required'])} (gap {n['gap']})")

In [ ]:
print('90-DAY TRANSITION PLAN\n')
rm = top['roadmap']
for phase_key, title in [('days_1_30', 'Days 1-30'), ('days_31_60', 'Days 31-60'), ('days_61_90', 'Days 61-90')]:
    phase = rm[phase_key]
    print(f"{title} - {phase['theme']}")
    for f in phase['focus']:
        print(f"    - {f}")
    print()

print('RECOMMENDED PORTFOLIO PROJECT')
print(f"  {top['portfolio_project']}")

## 6. Why am I being recommended this role?

The killer feature: every score is explainable. Here's the exact calculation - no black box.

In [ ]:
import json
print('TRANSITION SCORE breakdown')
print(json.dumps(top['transition_detail']['explanation'], indent=2))
print('\nFUTURE FIT breakdown')
print(json.dumps(top['future_fit_detail']['explanation'], indent=2))
print('\nMarket signals for this role')
print(json.dumps(top['market'], indent=2))

---
# Feature 2: Job Description AI-Exposure Analyzer

Paste any company's job description. The analyzer estimates an **AI takeover %**, lists the **actions AI can take over**, and the **actions that remain human-critical**. Same task-level engine, pointed at external text.

In [ ]:
job_description = '''
Senior Data Analyst - FinTech

Responsibilities:
- Build and maintain weekly and monthly reporting dashboards in Power BI
- Write SQL queries to extract, clean and transform data from the warehouse
- Perform exploratory analysis to investigate trends and root causes
- Partner with finance and product stakeholders to translate data into actionable insights
- Define requirements and scope new analytics initiatives with the leadership team
- Mentor junior analysts and lead the reporting roadmap
- Develop Python data pipelines and deploy them to the cloud
- Ensure compliance with data governance and privacy policies
'''

jd = analyze_jd(job_description)
print(f"AI takeover: {jd['ai_takeover_pct']}%  ({jd['tasks_detected']} tasks detected)")
print(jd['summary'])

In [ ]:
print('ACTIONS AI CAN TAKE OVER:')
for a in jd['ai_takeover_actions']:
    print(f"  - {a['task']} ({a['impact_label']}, exposure {a['exposure']})")

print('\nHUMAN + AI HYBRID:')
for a in jd['hybrid_actions']:
    print(f"  - {a['task']} ({a['impact_label']}, exposure {a['exposure']})")

print('\nACTIONS THAT REMAIN HUMAN-CRITICAL:')
for a in jd['human_critical_actions']:
    print(f"  - {a['task']} -> {a['human_value']}")

In [ ]:
df_jd = pd.DataFrame([
    {'Task': r['task'], 'Impact': r['impact_label'], 'Exposure': r['exposure'],
     'Matched text': r['matched_text'][:70]}
    for r in jd['breakdown']
])
df_jd

---
## Optional: richer narratives with an LLM (not required)

The engine is fully deterministic and free to run. If you *want* nicer prose, plug in your own model. This cell stays inert unless you set an API key, so the notebook remains free and open to run.

The scoring **stays deterministic** - the LLM only rewrites the numbers the engine produced. It never invents the recommendation.

In Colab you can store your key via `from google.colab import userdata` or `os.environ['OPENAI_API_KEY'] = '...'`.

In [ ]:
def narrate_with_llm(result: dict) -> str:
    """Optional. Requires `pip install openai` and OPENAI_API_KEY in the env.
    Returns a friendly summary built ONLY from the engine's deterministic output."""
    import os
    if not os.environ.get('OPENAI_API_KEY'):
        return '(LLM disabled: no OPENAI_API_KEY set. The engine output above is complete on its own.)'
    try:
        from openai import OpenAI
    except ImportError:
        return '(Install the optional dependency: pip install openai)'
    client = OpenAI()
    facts = {
        'current_role': result['current_role_match'],
        'ai_transformation_score': result['ai_transformation']['score'],
        'top_recommendations': [
            {'role': r['title'], 'transition': r['transition_score'], 'future_fit': r['future_fit'],
             'needs': [n['label'] for n in r['skill_gap']['need']]}
            for r in result['recommendations'][:3]
        ],
    }
    prompt = ('You are a career coach. Using ONLY these computed facts, write an encouraging '
              '150-word summary. Do not invent numbers or roles.\n\n' + str(facts))
    resp = client.chat.completions.create(
        model='gpt-4o-mini', messages=[{'role': 'user', 'content': prompt}])
    return resp.choices[0].message.content

print(narrate_with_llm(result))

---
## Notes & next steps

- **Data:** the bundled dataset is a curated *seed*. Swap in [O*NET](https://www.onetcenter.org/) (tasks/skills), [BLS](https://www.bls.gov/emp/) (growth/demand), and an open job-postings dataset (market demand / remote share) - keep the same JSON schema and the engine works unchanged.
- **Web app:** run `streamlit run app.py` from the GitHub repo, or host free on Streamlit Community Cloud.
- **Thesis:** *career evolution, not job replacement* - every recommendation shows its evidence.